<a href="https://colab.research.google.com/github/HelgaIngridHochbauer/Synthetic-data-generation/blob/main/Synth_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CompVis/stable-diffusion-v1-4

In [ ]:
import os
import random
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

def load_pipeline():
    """Load the Stable Diffusion pipeline with automatic device selection."""
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")
        pipe = pipe.to(device)

        # Disable the NSFW safety checker
        def dummy_safety_checker(images, clip_input, **kwargs):
            return images, [False] * len(images)

        pipe.safety_checker = dummy_safety_checker  # Apply the dummy safety checker
        return pipe, device
    except Exception as e:
        print(f"Error loading pipeline: {e}")
        raise RuntimeError("Failed to initialize Stable Diffusion pipeline.") from e

def load_images_from_folder(folder_path, max_images=None):
    """Load and preprocess all images from a folder."""
    try:
        image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
        if max_images:
            image_files = image_files[:max_images]  # Optionally limit the number of images
        images = []
        for file in image_files:
            try:
                img = Image.open(file).convert("RGB")
                img = img.resize((512, 512))  # Resize to 512x512 to match Stable Diffusion expectations
                images.append(img)
            except Exception as e:
                print(f"Error loading image {file}: {e}")
        if not images:
            raise ValueError(f"No valid images found in folder: {folder_path}")
        return images
    except Exception as e:
        print(f"Error loading images: {e}")
        raise

def save_generated_images(images, output_folder_path):
    """Save generated images to the specified folder with unique filenames."""
    os.makedirs(output_folder_path, exist_ok=True)
    for idx, generated_image in enumerate(images):
        output_file_name = os.path.join(output_folder_path, f"img_{idx + 1}.jpg")
        try:
            # Save the image with a unique filename (based on index)
            generated_image.save(output_file_name)
            print(f"Generated image saved as: {output_file_name}")
        except Exception as e:
            print(f"Error saving image {idx + 1}: {e}")

def process_images(pipe, images, prompt, output_folder_path):
    """Process images individually to avoid batch processing errors."""
    generated_images = []  # To hold the generated images before saving
    for idx, image in enumerate(images):
        try:
            # Process the image
            print(f"Processing image {idx + 1}/{len(images)}")
            result = pipe(
                prompt=prompt,
                image=image,
                strength=0.263, #263
                guidance_scale=3.45 #3.45
            )

            # Add generated image to list
            generated_images.append(result.images[0])  # Assuming the result is a list of images

        except Exception as e:
            print(f"Error during image generation for image {idx + 1}: {e}")

    # After all images are processed, save them
    save_generated_images(generated_images, output_folder_path)

def main():
    input_folder_path = "/content/drive/MyDrive/churches"  # Replace with your input folder
    output_folder_path = "/content/drive/MyDrive/New_Synth_Images/Case3"  # Folder to save output images
    base_prompt = "Satellite photography, vertical top-down view of a traditional Romanian Orthodox church,"
    variations = [
    "white stone walls, dark grey pointed shingle roofs, triconch architecture,",
    "surrounded by a small grassy courtyard and a stone perimeter wall, sharp shadows,",
    "wooden church nestled in the Carpathian mountains, steep spire, dark wood textures,"
    "orthophoto style, sharp focus, intricate roof patterns, vibrant natural colors, highly detailed landscape"

    ]
    prompt = f"{base_prompt} {random.choice(variations)}"

    # Load pipeline
    try:
        pipe, device = load_pipeline()
    except RuntimeError:
        print("Pipeline initialization failed. Exiting.")
        return

    # Load images
    try:
        images = load_images_from_folder(input_folder_path, max_images=200)  # Limit to 100 images
    except ValueError as e:
        print(e)
        return

    # Randomly shuffle the images to ensure the selection is different each time
    random.shuffle(images)

    # Optionally, select a subset of images for blending or generation
    num_images_to_use = 200 # You can change this number based on your requirements
    selected_images = images[:num_images_to_use]

    # Process images individually
    process_images(pipe, selected_images, prompt, output_folder_path)

if __name__ == "__main__":
    main()


#CompVis/stable-diffusion-v1-4 with negative prompt

In [ ]:
import os
import random
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

def load_pipeline():
    """Load the Stable Diffusion pipeline with automatic device selection."""
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        # Using the standard v1-4 model as per your original code
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")
        pipe = pipe.to(device)

        # Disable the NSFW safety checker
        def dummy_safety_checker(images, clip_input, **kwargs):
            return images, [False] * len(images)

        pipe.safety_checker = dummy_safety_checker
        return pipe, device
    except Exception as e:
        print(f"Error loading pipeline: {e}")
        raise RuntimeError("Failed to initialize Stable Diffusion pipeline.") from e

def load_images_from_folder(folder_path, max_images=None):
    """Load and preprocess all images from a folder."""
    try:
        image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
        if max_images:
            image_files = image_files[:max_images]
        images = []
        for file in image_files:
            try:
                img = Image.open(file).convert("RGB")
                img = img.resize((512, 512))
                images.append(img)
            except Exception as e:
                print(f"Error loading image {file}: {e}")
        if not images:
            raise ValueError(f"No valid images found in folder: {folder_path}")
        return images
    except Exception as e:
        print(f"Error loading images: {e}")
        raise

def save_generated_images(images, output_folder_path):
    """Save generated images to the specified folder with unique filenames."""
    os.makedirs(output_folder_path, exist_ok=True)
    for idx, generated_image in enumerate(images):
        output_file_name = os.path.join(output_folder_path, f"img6{idx + 1}.jpg")
        try:
            generated_image.save(output_file_name)
            print(f"Saved: {output_file_name}")
        except Exception as e:
            print(f"Error saving image {idx + 1}: {e}")

def process_images(pipe, images, prompt, negative_prompt, output_folder_path):
    """Process images individually with negative prompt support."""
    generated_images = []
    for idx, image in enumerate(images):
        try:
            print(f"Processing image {idx + 1}/{len(images)}")
            # The pipe call now includes the negative_prompt argument
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=image,
                strength=0.3,
                guidance_scale=3.43
            )
            generated_images.append(result.images[0])

        except Exception as e:
            print(f"Error during image generation for image {idx + 1}: {e}")

    # Save all successful results
    save_generated_images(generated_images, output_folder_path)

def main():
    # --- Paths Configuration ---
    input_folder_path = "/content/drive/MyDrive/New_Synth_Images/Case3"
    output_folder_path = "/content/drive/MyDrive/New_Synth_Images/case3_extra"

    # --- Prompt Configuration ---
    base_prompt = "A high-resolution satellite top-down view of a church, "
    variations = [
        "traditional, white stone walls, dark grey pointed shingle roofs, triconch architecture,",
        "surrounded by a small grassy courtyard and a stone perimeter wall",
        "centered in a rugged desert landscape with realistic terrain textures."

    ]

    # Selecting a random style for this batch
    prompt = f"{base_prompt} {random.choice(variations)}"

    # The Negative Prompt: filters out things you DON'T want
    negative_prompt = (
        "low resolution, blurry, distorted architecture, text, watermark, signature, "
        "cars, modern buildings, fisheye lens, 3d render, cartoon, painting, "
        "sideways view, horizon line, person, crowded, noisy, grainy"
    )

    # --- Execution ---
    try:
        pipe, device = load_pipeline()
    except RuntimeError:
        return

    try:
        images = load_images_from_folder(input_folder_path, max_images=2000)
        random.shuffle(images)
        selected_images = images[:2000]
    except ValueError as e:
        print(e)
        return

    # Start processing with the new negative prompt parameter
    process_images(pipe, selected_images, prompt, negative_prompt, output_folder_path)

if __name__ == "__main__":
    main()

#runwayml/stable-diffusion-v1-5

In [ ]:
import os
import torch
from PIL import Image, ImageOps
from diffusers import StableDiffusionImg2ImgPipeline

# --- CONFIGURATION ---
INPUT_DIR = 'filtered_output'
SYNTHETIC_DIR = 'synthetic_data'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "runwayml/stable-diffusion-v1-5" # Good for general synthetic data

if not os.path.exists(SYNTHETIC_DIR):
    os.makedirs(SYNTHETIC_DIR)

# --- LOAD MODEL ---
# Using float16 for faster performance and lower memory on Colab
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

def preprocess_image(img_path, size=(512, 512)):
    """Resizes and pads image to maintain aspect ratio without stretching."""
    image = Image.open(img_path).convert("RGB")
    image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
    return image

def generate_synthetic_data(prompt, strength=0.75, guidance_scale=7.5):
    exts = ('.jpg', '.jpeg', '.png')
    images = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(exts)]

    print(f"Generating synthetic versions for {len(images)} images...")

    for filename in images:
        init_image = preprocess_image(os.path.join(INPUT_DIR, filename))

        # Generate new image based on the original structure
        # strength: 0.0 = same as original, 1.0 = completely new image
        with torch.autocast(DEVICE):
            gen_image = pipe(
                prompt=prompt,
                image=init_image,
                strength=strength,
                guidance_scale=guidance_scale
            ).images[0]

        gen_image.save(os.path.join(SYNTHETIC_DIR, f"syn_{filename}"))
        print(f"  [+] Generated: syn_{filename}")

# --- EXECUTION ---

MY_PROMPT = "a high quality, sharp, detailed realistic photo of an thomb"
generate_synthetic_data(MY_PROMPT)

#controlnet-canny-sdxl-1.0 with SG161222/RealVisXL_V4.0

In [ ]:
import os
import torch
from PIL import Image
from diffusers import StableDiffusionXLControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler

# 1. SET YOUR PATHS
input_folder = "/content/drive/MyDrive/churches"
output_folder = "/content/drive/MyDrive/New_Synth_Images/Case3"
os.makedirs(output_folder, exist_ok=True)

# 2. LOAD MODELS
# We use ControlNet to ensure the walls/streets don't move
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16
)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
    "SG161222/RealVisXL_V4.0", # The recommended photoreal model
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# 3. SETTINGS
prompt = "Realistic satelitary image with a traditional chirch seen from above"
negative_prompt = "3D render, new construction, vibrant colors, pristine, people, distorted geometry, extra buildings."

# 4. BATCH PROCESS
for filename in os.listdir(input_folder):
    if filename.endswith((".png", ".jpg", ".jpeg")):
        img_path = os.path.join(input_folder, filename)
        init_image = Image.open(img_path).convert("RGB")

        # We use the original image as the 'control' to lock geometry
        # Low denoising_strength (0.3-0.4) adds noise/fading without changing layout
        output = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=init_image,
            control_image=init_image,
            strength=0.35,          # Controls how much 'degradation' is added
            controlnet_conditioning_scale=0.8, # How strictly to follow the lines
            num_inference_steps=30
        ).images[0]

        output.save(os.path.join(output_folder, f"weathered_{filename}"))
        print(f"Processed: {filename}")

print("Batch processing complete!")